## Example 1: Basic Grouped Aggregation Operations

In [ ]:
import flox
import flox.xarray
import numpy as np
import xarray as xr

# Example 1: Basic grouped aggregation operations
print("Example 1: Basic Grouped Aggregation Operations")

try:
    # Create sample xarray Dataset
    np.random.seed(42)
    
    # Create sample data with multiple dimensions
    time = np.arange(100)
    locations = ['A', 'B', 'C', 'D', 'E']
    
    # Create temperature data with seasonal pattern
    temperature = 20 + 10 * np.sin(2 * np.pi * time / 365.25) + np.random.normal(0, 2, (len(locations), len(time)))
    
    # Create precipitation data 
    precipitation = np.maximum(0, 5 + 3 * np.cos(2 * np.pi * time / 365.25) + np.random.normal(0, 1, (len(locations), len(time))))
    
    # Create xarray Dataset
    ds = xr.Dataset({
        'temperature': (['location', 'time'], temperature),
        'precipitation': (['location', 'time'], precipitation)
    }, coords={
        'location': locations,
        'time': time,
        'season': ('time', np.where(time % 4 == 0, 'winter',
                                  np.where(time % 4 == 1, 'spring', 
                                         np.where(time % 4 == 2, 'summer', 'fall'))))
    })
    
    print(f"Created dataset with shape: {ds.dims}")
    print(f"Variables: {list(ds.data_vars)}")
    print(f"Coordinates: {list(ds.coords)}")
    
    # Basic groupby operations with flox
    print(f"\nBasic flox groupby operations:")
    
    # Group by season and calculate mean temperature. flox.groupby_reduce is the NUMPY-level API
    # (axis=, no dim=); labelled xarray objects go through flox.xarray.xarray_reduce (flox 0.11)
    seasonal_temp_mean = flox.xarray.xarray_reduce(
        ds.temperature, 
        ds.season, 
        func="mean",
        dim="time"
    )
    
    print(f"Seasonal temperature means:")
    for season in seasonal_temp_mean.season:
        temp_vals = seasonal_temp_mean.sel(season=season).values
        print(f"  {season.values}: {temp_vals.mean():.2f}°C (±{temp_vals.std():.2f})")
    
    # Group by season and calculate total precipitation
    seasonal_precip_sum = flox.xarray.xarray_reduce(
        ds.precipitation,
        ds.season,
        func="sum", 
        dim="time"
    )
    
    # Cross-check flox against a plain numpy reduction for one group
    winter = ds.season.values == "winter"
    np.testing.assert_allclose(
        seasonal_temp_mean.sel(season="winter").values, ds.temperature.values[:, winter].mean(axis=1)
    )
    
    print(f"\nSeasonal precipitation totals:")
    for season in seasonal_precip_sum.season:
        precip_vals = seasonal_precip_sum.sel(season=season).values
        print(f"  {season.values}: {precip_vals.mean():.1f}mm (±{precip_vals.std():.1f})")
    
    # Multiple aggregation functions
    print(f"\nMultiple aggregation functions:")
    
    # Calculate multiple statistics for temperature
    # xarray_reduce takes ONE func per call; stack the four along a new `func` dimension
    stat_funcs = ["mean", "std", "min", "max"]
    temp_stats = xr.concat(
        [flox.xarray.xarray_reduce(ds.temperature, ds.season, func=f, dim="time") for f in stat_funcs],
        dim="func",
    ).assign_coords(func=stat_funcs)
    
    print(f"Temperature statistics by season:")
    for season in temp_stats.season:
        season_data = temp_stats.sel(season=season)
        mean_vals = season_data.sel(func="mean").values
        std_vals = season_data.sel(func="std").values
        min_vals = season_data.sel(func="min").values  
        max_vals = season_data.sel(func="max").values
        
        print(f"  {season.values}:")
        print(f"    Mean: {mean_vals.mean():.1f}°C, Std: {std_vals.mean():.1f}°C")
        print(f"    Range: {min_vals.min():.1f}°C to {max_vals.max():.1f}°C")
    
    print("Basic grouped aggregation operations completed successfully!")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback demonstration
    print("Demonstrating grouped operations concepts:")
    
    # Simple numpy-based example
    data = np.random.normal(20, 5, 100)
    groups = np.repeat(['A', 'B', 'C', 'D'], 25)
    
    print("Fallback grouping with numpy:")
    unique_groups = np.unique(groups)
    for group in unique_groups:
        group_data = data[groups == group]
        print(f"  Group {group}: mean={group_data.mean():.1f}, std={group_data.std():.1f}")
        
    concepts = [
        "Grouped aggregation: Apply functions to data grouped by categories",
        "Multiple statistics: Calculate mean, std, min, max simultaneously", 
        "Dimensional operations: Group along specific dimensions",
        "Efficient computation: Optimized algorithms for large datasets",
        "xarray integration: Works seamlessly with labeled arrays"
    ]
    
    for concept in concepts:
        print(f"  - {concept}")
    raise

## Example 2: NumPy-level groupby_reduce with NaN-aware reductions

In [ ]:
import flox
import numpy as np

# Example 2: flox.groupby_reduce works on plain NumPy arrays: reduce along an axis, by label.
rng = np.random.default_rng(0)
values = rng.normal(size=(3, 12))
values[0, 3] = np.nan
labels = np.array([0, 1, 2] * 4)

result, groups = flox.groupby_reduce(values, labels, func="nanmean", axis=-1)
print("groups:", groups, "| result shape:", result.shape)
expected = np.stack([np.nanmean(values[:, labels == g], axis=-1) for g in groups], axis=-1)
np.testing.assert_allclose(result, expected)
assert not np.isnan(result).any()  # nanmean skipped the missing value

# expected_groups adds groups that have no members; fill_value says what they hold.
padded, padded_groups = flox.groupby_reduce(
    values, labels, func="nansum", expected_groups=np.array([0, 1, 2, 3]), axis=-1, fill_value=0
)
assert list(padded_groups) == [0, 1, 2, 3]
assert padded.shape == (3, 4) and (padded[:, 3] == 0).all()
print("Example 2 finished")

## Example 3: Binning continuous values with isbin

In [ ]:
import flox
import numpy as np
import pandas as pd

# Example 3: isbin=True treats expected_groups as bin edges (right-closed, like pandas.cut).
rng = np.random.default_rng(1)
values = rng.uniform(0, 10, size=200)
edges = np.array([0, 2.5, 5, 7.5, 10])

counts, bins = flox.groupby_reduce(np.ones_like(values), values, func="count",
                                   expected_groups=edges, isbin=True)
print("bins:", [str(b) for b in bins])
print("counts:", counts)
reference = pd.Series(pd.cut(values, edges)).value_counts(sort=False).to_numpy()
np.testing.assert_array_equal(counts, reference)
assert counts.sum() == len(values)

means, _ = flox.groupby_reduce(values, values, func="mean", expected_groups=edges, isbin=True)
assert all(lo < m <= hi for m, lo, hi in zip(means, edges[:-1], edges[1:]))  # each mean lies inside its bin
print("Example 3 finished")

## Example 4: The numpy and flox engines agree

In [ ]:
import flox
import numpy as np

# Example 4: engine="numpy" and engine="flox" are two implementations of the same reductions.
rng = np.random.default_rng(2)
values = rng.normal(size=(4, 30))
values[1, ::7] = np.nan
labels = rng.integers(0, 5, size=30)

for func in ["sum", "nansum", "nanmean", "nanmax", "nanmin", "count"]:
    by_numpy, groups_numpy = flox.groupby_reduce(values, labels, func=func, axis=-1, engine="numpy")
    by_flox, groups_flox = flox.groupby_reduce(values, labels, func=func, axis=-1, engine="flox")
    np.testing.assert_array_equal(groups_numpy, groups_flox)
    np.testing.assert_allclose(by_numpy, by_flox, equal_nan=True)
    print(f"  {func}: engines agree")
print("Example 4 finished")

## Example 5: Grouping by two variables at once

In [ ]:
import flox
import flox.xarray
import numpy as np
import pandas as pd
import xarray as xr

# Example 5: pass several label arrays to group by their combinations; the result gets one axis per grouper.
values = np.arange(12.0)
kind = np.array(list("aabbaabbaabb"))
period = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])

totals, kinds, periods = flox.groupby_reduce(values, kind, period, func="sum")
print("kinds:", kinds, "| periods:", periods)
print(totals)
reference = pd.DataFrame({"v": values, "kind": kind, "period": period}).groupby(["kind", "period"]).v.sum().unstack()
np.testing.assert_allclose(totals, reference.to_numpy())

# The same reduction on a labelled DataArray, grouping by two coordinates.
da = xr.DataArray(values, dims="t", coords={"kind": ("t", kind), "period": ("t", period)})
labelled = flox.xarray.xarray_reduce(da, "kind", "period", func="sum")
assert labelled.dims == ("kind", "period")
np.testing.assert_allclose(labelled.values, totals)
print("Example 5 finished")